# 02 — REDSEA spillover correction and selected expression

This notebook inspects the production `redsea` and `expression` artifacts. REDSEA owns qptiff rasterization, contact correction, compartment output, alignment checks, and fail-fast donor execution. The marker registry then selects exactly one authoritative intensity per marker. No compensation or source-selection math is reimplemented here.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from phenocycler.artifacts import StageManifest
from phenocycler.config import load_config
from phenocycler.pipeline import RunContext, resolve_run_context, run_stage, status

CONFIG_PATH = None
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
status_code = status(context)
print(f"status return code: {status_code}")

## Optional production execution

`redsea` requires current ingest and geometry manifests. `expression` requires current ingest, geometry, REDSEA, and marker-registry fingerprints. Existing valid stages are checked rather than recomputed.

In [ ]:
RUN_STAGES = False

if RUN_STAGES:
    for stage_name in ("redsea", "expression"):
        run_stage(context, stage_name)
else:
    print("Inspection only. Set RUN_STAGES=True to run production REDSEA and expression selection.")

In [ ]:
manifest_rows = []
for stage_name in ("redsea", "expression"):
    path = context.stage_manifest_path(stage_name)
    if path.exists():
        manifest = StageManifest.read_json(path)
        manifest_rows.append({
            "stage": stage_name,
            "method_version": manifest.method_version,
            "donors": len(manifest.completed_donors),
            "rows": manifest.output.total_rows,
            "columns": len(manifest.output.schema),
            "schema": manifest.output.schema_sha256[:12],
            "objects": manifest.output.object_id_sha256[:12],
            "content": manifest.content_id[:12],
        })
display(pd.DataFrame(manifest_rows))

## Authoritative marker source

The registry—not notebook column guessing—declares each marker's compartment and whether its value is REDSEA-corrected, REDSEA passthrough, or uncompensated. Markers absent from a donor's acquisition panel remain null with an availability reason.

In [ ]:
registry_rows = [
    {
        "marker": marker.name,
        "kind": marker.kind,
        "compartment": marker.compartment,
        "spillover_policy": marker.spillover_policy,
        "measurement_column": marker.measurement_column,
        "reference": marker.reference,
        "calibration_status": marker.calibration_status,
    }
    for marker in context.registry.markers
]
display(pd.DataFrame(registry_rows))

## Inspect REDSEA and selected expression

The REDSEA table is compartment-resolved. The selected-expression table is intentionally narrow: one value per registered marker plus cell identity, spatial context, and geometry eligibility. A corrected value may increase or decrease because the configured REDSEA operator includes both boundary reinforcement and neighbor-spillover subtraction.

In [ ]:
DONOR = context.donors[0]
required = (context.stage_manifest_path("redsea"), context.stage_manifest_path("expression"))
if all(path.exists() for path in required):
    redsea_glob = (context.config.redsea_dir / f"donor_id={DONOR}" / "*.parquet").as_posix()
    expression_glob = (context.config.selected_expression_dir / f"donor_id={DONOR}" / "*.parquet").as_posix()
    connection = duckdb.connect()
    connection.execute(f"SET threads={int(context.config.duckdb_threads)}")
    redsea_cursor = connection.execute("SELECT * FROM read_parquet(?) LIMIT 0", [redsea_glob])
    redsea_columns = [item[0] for item in redsea_cursor.description]
    expression_cursor = connection.execute("SELECT * FROM read_parquet(?) LIMIT 0", [expression_glob])
    expression_columns = [item[0] for item in expression_cursor.description]
    redsea_measurements = [column for column in redsea_columns if "__" in column]
    quote = lambda column: '"' + column.replace('"', '""') + '"'
    redsea_projection = ["object_id", *redsea_measurements[:10]]
    redsea_preview = connection.execute(
        f"SELECT {', '.join(map(quote, redsea_projection))} FROM read_parquet(?) "
        "ORDER BY hash(CAST(object_id AS VARCHAR)) LIMIT 10", [redsea_glob],
    ).fetchdf()
    redsea_count = connection.execute("SELECT count(*) FROM read_parquet(?)", [redsea_glob]).fetchone()[0]
    expression_count = connection.execute("SELECT count(*) FROM read_parquet(?)", [expression_glob]).fetchone()[0]
    print(f"donor {DONOR}: REDSEA rows={redsea_count:,}, selected-expression rows={expression_count:,}")
    display(redsea_preview)
    context_columns = [
        column for column in (
            "donor_id", "object_id", "image", "cell_region",
            "qc_analysis_eligible", "qc_estimation_eligible"
        ) if column in expression_columns
    ]
    marker_columns = [marker.name for marker in context.registry.markers if marker.name in expression_columns]
    expression_projection = [*context_columns, *marker_columns[:8]]
    expression_preview = connection.execute(
        f"SELECT {', '.join(map(quote, expression_projection))} FROM read_parquet(?) "
        "ORDER BY hash(CAST(object_id AS VARCHAR)) LIMIT 10", [expression_glob],
    ).fetchdf()
    display(expression_preview)
else:
    print("REDSEA/expression artifacts are not complete yet.")

In [ ]:
availability_path = context.config.audit_dir / "expression_availability.parquet"
if availability_path.exists():
    availability = pd.read_parquet(availability_path)
    display(
        availability.groupby(["available", "reason"], dropna=False).size()
        .rename("donor_markers").to_frame()
    )
    display(availability.loc[availability["donor_id"].astype(str).eq(DONOR)].head(20))
else:
    print("Expression availability audit is not present.")

## Panel and source-availability decision plot

A missing marker is unavailable, never zero. This heatmap exposes donor/marker gaps before calibration or typing. Unexpected gaps block the handoff; expected panel absence remains explicit downstream.

In [ ]:
if 'availability' in locals() and not availability.empty:
    availability_plot = availability.copy()
    availability_plot['donor_id'] = availability_plot['donor_id'].astype(str)
    availability_matrix = (
        availability_plot.pivot(index='donor_id', columns='marker', values='available')
        .reindex(index=context.donors, columns=[marker.name for marker in context.registry.markers])
    )
    values = availability_matrix.astype('float').to_numpy()
    fig, ax = plt.subplots(
        figsize=(max(13, 0.38 * len(availability_matrix.columns)), max(6, 0.30 * len(availability_matrix))),
        constrained_layout=True,
    )
    image = ax.imshow(np.ma.masked_invalid(values), vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(np.arange(len(availability_matrix.columns)), labels=availability_matrix.columns, rotation=60, ha='right')
    ax.set_yticks(np.arange(len(availability_matrix.index)), labels=availability_matrix.index)
    ax.set_xlabel('Marker')
    ax.set_ylabel('Donor')
    ax.set_title('Authoritative marker-source availability')
    fig.colorbar(image, ax=ax, ticks=[0, 1], label='0 unavailable · 1 available')
    plt.show()
else:
    print("Expression availability audit is not available to plot.")

## REDSEA candidate preservation plots

Select one corrected marker below. Raw-versus-corrected and mutually exclusive target/reference plots reveal the magnitude and direction of correction on a deterministic cell sample. They are diagnostics, not sufficient acceptance tests. A REDSEA parameter set is accepted only when neighbor-associated and implausible co-positive signal falls **and** independently reviewed isolated/dim positives are retained; double-positive reduction alone can hide false-negative inflation. Persist those preservation metrics in a refinement audit before promotion.

In [ ]:
corrected_specs = [
    marker for marker in context.registry.active_markers
    if marker.spillover_policy == 'redsea_corrected' and marker.reference is not None
]
MARKER = corrected_specs[0].name if corrected_specs else None
REDSEA_PLOT_CELLS = 50_000

if MARKER is None or not all(path.exists() for path in required):
    print("No completed corrected marker is available to plot.")
else:
    marker_spec = context.registry.marker(MARKER)
    reference_spec = context.registry.marker(marker_spec.reference)
    cells_glob = (context.config.cells_dir / f"donor_id={DONOR}" / "*.parquet").as_posix()
    target_corrected = f"{marker_spec.compartment}__{marker_spec.name}"
    reference_corrected = f"{reference_spec.compartment}__{reference_spec.name}"
    required_columns = {MARKER, marker_spec.reference, target_corrected, reference_corrected}
    available_columns = set(redsea_columns) | set(
        item[0] for item in connection.execute("SELECT * FROM read_parquet(?) LIMIT 0", [cells_glob]).description
    )
    missing = sorted(required_columns.difference(available_columns))
    if missing:
        raise KeyError(f"REDSEA diagnostic columns are missing: {missing}")
    q = quote
    correction_sample = connection.execute(
        f"""
        SELECT c.object_id, c.{q(MARKER)} AS target_raw, r.{q(target_corrected)} AS target_corrected,
               c.{q(marker_spec.reference)} AS reference_raw,
               r.{q(reference_corrected)} AS reference_corrected
        FROM read_parquet(?) c
        JOIN read_parquet(?) r USING (object_id, donor_id)
        WHERE isfinite(c.{q(MARKER)}) AND isfinite(r.{q(target_corrected)})
          AND isfinite(c.{q(marker_spec.reference)}) AND isfinite(r.{q(reference_corrected)})
        ORDER BY sha256(CAST(c.object_id AS VARCHAR) || ?)
        LIMIT ?
        """,
        [cells_glob, redsea_glob, f'{context.run_id}|{DONOR}|{MARKER}', REDSEA_PLOT_CELLS],
    ).fetchdf()
    for column in ('target_raw', 'target_corrected', 'reference_raw', 'reference_corrected'):
        correction_sample[column] = pd.to_numeric(correction_sample[column], errors='coerce').clip(lower=0)
    correction_sample['target_change_log2'] = np.log2(1 + correction_sample['target_corrected']) - np.log2(1 + correction_sample['target_raw'])

    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), constrained_layout=True)
    target_raw_log = np.log1p(correction_sample['target_raw'])
    target_corrected_log = np.log1p(correction_sample['target_corrected'])
    low = float(min(target_raw_log.quantile(.005), target_corrected_log.quantile(.005)))
    high = float(max(target_raw_log.quantile(.995), target_corrected_log.quantile(.995)))
    axes[0].hexbin(target_raw_log, target_corrected_log, gridsize=70, mincnt=1, bins='log', cmap='viridis')
    axes[0].plot([low, high], [low, high], '--', color='white', linewidth=1)
    axes[0].set_xlim(low, high); axes[0].set_ylim(low, high)
    axes[0].set_xlabel(f'Raw log1p {MARKER}')
    axes[0].set_ylabel(f'Corrected log1p {MARKER}')
    axes[0].set_title('Per-cell correction')

    change = correction_sample['target_change_log2'].replace([np.inf, -np.inf], np.nan).dropna()
    change_low, change_high = change.quantile([.005, .995])
    axes[1].hist(change.clip(change_low, change_high), bins=100, color='#4c78a8')
    axes[1].axvline(0, color='black', linewidth=1)
    axes[1].set_xlabel('log2(1 + corrected) − log2(1 + raw)')
    axes[1].set_ylabel('Sampled cells')
    axes[1].set_title('Correction direction and magnitude')

    before_x = np.log1p(correction_sample['reference_raw'])
    before_y = np.log1p(correction_sample['target_raw'])
    after_x = np.log1p(correction_sample['reference_corrected'])
    after_y = np.log1p(correction_sample['target_corrected'])
    axes[2].scatter(before_x, before_y, s=3, alpha=.08, color='#e45756', label='raw')
    axes[2].scatter(after_x, after_y, s=3, alpha=.08, color='#4c78a8', label='corrected')
    axes[2].set_xlabel(f'log1p {marker_spec.reference} (mutually exclusive reference)')
    axes[2].set_ylabel(f'log1p {MARKER}')
    axes[2].set_title('Reference/target separation — descriptive')
    axes[2].legend(markerscale=3)
    for ax in axes:
        ax.grid(alpha=.15)
    fig.suptitle(f'Donor {DONOR} · {MARKER} REDSEA diagnostic sample (n={len(correction_sample):,})')
    plt.show()
    display(correction_sample[['target_raw', 'target_corrected', 'target_change_log2']].describe(percentiles=[.01, .1, .5, .9, .99]))

## Handoff

Proceed to calibration only when `redsea` and `expression` are `CURRENT` and the availability audit agrees with the acquisition panels. The selected-expression artifact is the sole intensity input to reference-control selection and donor-marker calibration.